# THỐNG KÊ MÁY TÍNH VÀ ỨNG DỤNG (ĐTTX)

**Khoa Công nghệ Thông tin - ĐH Khoa học Tự nhiên TP. HCM ([fit@hcmus](https://www.fit.hcmus.edu.vn/))**

*Giảng viên: Vũ Quốc Hoàng (vqhoang@fit.hcmus.edu.vn)*

# BÀI 4 - XỬ LÝ BẢNG DỮ LIỆU VỚI PANDAS (Phần 3)

**Nội dung**

* [Các thao tác vector hóa trên chuỗi](#chuoi)
* [Ghép bảng đơn giản với `concat`](#concat)
* [Kết (nối, trộn) các bảng dữ liệu](#merge)
  * [Các thể loại kết](#loai_ket)
  * [Xác định cách giữ lại các dòng trong bảng kết quả](#ket_qua)
  * [Xác định khóa để kết bảng](#key)

**Tài liệu tham khảo**

* Chương 18, 19, 22 [Python Data Science Handbook (Jake VanderPlas)](https://www.amazon.com/Python-Data-Science-Handbook-Essential-dp-1098121228/dp/1098121228/)

In [1]:
import numpy as np
import pandas as pd

## <a name="chuoi"/>Các thao tác vector hóa trên chuỗi

Ta đã thấy thao tác vector hóa trên các mảng số trong NumPy hay các bảng số (`Series`, `DataFrame`) trong pandas.

In [2]:
sr = pd.Series(np.arange(5))
print(sr)
print(sr**2)

0    0
1    1
2    2
3    3
4    4
dtype: int32
0     0
1     1
2     4
3     9
4    16
dtype: int32


Tương tự, pandas cũng hỗ trợ các thao tác vector hóa trên chuỗi rất hiệu quả bằng thuộc tính `str` của `Series` hay `DataFrame`.

In [3]:
data = ['peter', 'Paul', 'MARY', 'gUIDO']
[s.capitalize() for s in data]

['Peter', 'Paul', 'Mary', 'Guido']

In [4]:
sr = pd.Series(data)
sr

0    peter
1     Paul
2     MARY
3    gUIDO
dtype: object

In [5]:
sr.str.capitalize()

0    Peter
1     Paul
2     Mary
3    Guido
dtype: object

In [6]:
sr.str.len()

0    5
1    4
2    4
3    5
dtype: int64

In [7]:
sr.str[:3]

0    pet
1    Pau
2    MAR
3    gUI
dtype: object

In [8]:
sr.str.contains("a", case=False)

0    False
1     True
2     True
3    False
dtype: bool

In [9]:
sr[sr.str.contains("a", case=False)]

1    Paul
2    MARY
dtype: object

pandas cung cấp rất nhiều hàm tương tự, tham khảo: [Working with text data](https://pandas.pydata.org/docs/user_guide/text.html#method-summary).

**Bài tập**

1. Nạp lại bảng dữ liệu `Iris Data Set`

1. Dùng các thao tác trên chuỗi để loại bỏ tiền tố Iris- trong nhãn các lớp (cột class), ví dụ: Iris-setosa thay bằng setosa

1. Thay cột class là chuỗi thành cột số mã cho nhãn lớp với 0 là Iris-setosa, 1 là Iris-versicolor, 2 là Iris-virginica

In [10]:
# TODO


## <a name="concat"/>Ghép bảng đơn giản với `concat`

Tương tự như `np.concatenate`, `pd.concat` cho phép ghép bảng (`Series`, `DataFrame`) với nhiều lựa chọn khác nhau. Hơn nữa `pd.concat` có xử lý các chỉ mục (`index`, `columns`).

In [11]:
ser1 = pd.Series(np.random.randint(0, 10, 3), index=list("abc"))
ser2 = pd.Series(np.random.randint(0, 10, 3), index=list("def"))
print(ser1)
print(ser2)
print(pd.concat([ser1, ser2]))

a    8
b    9
c    6
dtype: int32
d    0
e    2
f    8
dtype: int32
a    8
b    9
c    6
d    0
e    2
f    8
dtype: int32


In [12]:
df1 = pd.DataFrame(np.random.randint(0, 10, (3, 2)), index=list("abc"), columns=list("AB"))
df1

,A,B
a,8,7
b,8,9
c,4,4


In [13]:
df2 = pd.DataFrame(np.random.randint(0, 10, (3, 2)), index=list("cbd"), columns=list("BC"))
df2

,B,C
c,4,6
b,1,2
d,6,2


Mặc định, việc ghép bảng được tiến hành theo dòng (ghép các dòng từ nhiều bảng thành 1 bảng lớn).

In [14]:
pd.concat([df1, df2])

,A,B,C
a,8.0,7,NaN
b,8.0,9,NaN
c,4.0,4,NaN
c,NaN,4,6.0
b,NaN,1,2.0
d,NaN,6,2.0


Việc ghép bảng được tiến hành theo cột (ghép các cột từ nhiều bảng thành 1 bảng lớn) với đối số `axis=1`

In [15]:
sr1 = pd.Series(np.random.randint(0, 10, 3), name="col1")
sr2 = pd.Series(np.random.rand(3), name="col2")
pd.concat([sr1, sr2], axis=1)

,col1,col2
0,4,0.365273
1,5,0.402109
2,8,0.244690


In [16]:
pd.concat([df1, df2], axis=1)

,A,B,B,C
a,8.0,7.0,NaN,NaN
b,8.0,9.0,1.0,2.0
c,4.0,4.0,4.0,6.0
d,NaN,NaN,6.0,2.0


Mặc định các chỉ mục trùng nhau (theo dòng hay cột) được giữ lại. Ta có thể cấm điều này với đối số `verify_integrity=True`.

In [17]:
try:
    pd.concat([df1, df2], axis=1, verify_integrity=True)
except Exception as e:
    print(e)

Indexes have overlapping values: Index(['B'], dtype='object')


Ta cũng có thể dùng "chỉ mục phân cấp" với tham số `keys`.

In [18]:
pd.concat([df1, df2], axis=1, keys=["df1", "df2"])

df1       df2     
     A    B    B    C
a  8.0  7.0  NaN  NaN
b  8.0  9.0  1.0  2.0
c  4.0  4.0  4.0  6.0
d  NaN  NaN  6.0  2.0

Ta cũng có thể bỏ qua chỉ mục ghi ghép với đối số `ignore_index=True`.

In [19]:
pd.concat([df1, df2], axis=1, ignore_index=True)

,0,1,2,3
a,8.0,7.0,NaN,NaN
b,8.0,9.0,1.0,2.0
c,4.0,4.0,4.0,6.0
d,NaN,NaN,6.0,2.0


Mặc định, khi ghép cột thì các dòng sẽ là hợp các dòng của các bảng và khi ghép dòng thì các cột sẽ là hợp các cột. Ta có thể thay đổi thành giao với `join="inner"`.

In [20]:
pd.concat([df1, df2], axis=1)

,A,B,B,C
a,8.0,7.0,NaN,NaN
b,8.0,9.0,1.0,2.0
c,4.0,4.0,4.0,6.0
d,NaN,NaN,6.0,2.0


In [21]:
pd.concat([df1, df2], axis=1, join="inner")

,A,B,B,C
b,8,9,1,2
c,4,4,4,6


In [22]:
pd.concat([df1, df2])

,A,B,C
a,8.0,7,NaN
b,8.0,9,NaN
c,4.0,4,NaN
c,NaN,4,6.0
b,NaN,1,2.0
d,NaN,6,2.0


In [23]:
pd.concat([df1, df2], join="inner")

,B
a,7
b,9
c,4
c,4
b,1
d,6


**Bài tập** Trên bảng dữ liệu [`Iris Data Set`](https://archive.ics.uci.edu/ml/datasets/iris)

1. Trích ra các bảng con cho từng giống hoa rồi ghép các bảng con trên thành 1 bảng và so sánh với bảng gốc

1. Trích ra 2 bảng con, 1 bảng gồm các đặc trưng và 1 bảng là nhãn lớp, rồi ghép 2 bảng con trên thành 1 bảng và so sánh với bảng gốc

In [24]:
# TODO


## <a name="merge"/>Kết (nối, trộn) các bảng dữ liệu

Tương tự các công cụ thao tác trên **cơ sở dữ liệu quan hệ** (realational database), pandas hỗ trợ các thao tác **kết** (combine) hay **nối, trộn** (join, merge) các bảng dữ liệu rất hiệu quả với `merge` và `join`.

### <a name="loai_ket"/>Các thể loại kết

Ví dụ, ta có 2 bảng điểm của các sinh viên, bảng điểm môn 1 và môn 2. Sinh viên được xác định qua **khóa** (key) là mã số sinh viên (MSSV).

In [25]:
score1 = pd.DataFrame({'MSSV': ['SV1', 'SV2', 'SV3'], 'Điểm1': [7, 5, 9]})
score1

,MSSV,Điểm1
0,SV1,7
1,SV2,5
2,SV3,9


In [26]:
score2 = pd.DataFrame({'MSSV': ['SV3', 'SV1', 'SV4'], 'Điểm2': [8, 4, 6]})
score2

,MSSV,Điểm2
0,SV3,8
1,SV1,4
2,SV4,6


Ta có thể dùng `merge` để kết 2 bảng này lại thành một bảng điểm dựa trên khóa là MSSV.

In [27]:
pd.merge(score1, score2)

,MSSV,Điểm1,Điểm2
0,SV1,7,4
1,SV3,9,8


Ví dụ, ta có thêm bảng thông tin sinh viên.

In [28]:
stu_infor = pd.DataFrame({'MSSV': ['SV1', 'SV2', 'SV3', 'SV4'], 'Tên sinh viên': ["An", "Bình", "Chương", "Dũng"]})
stu_infor

,MSSV,Tên sinh viên
0,SV1,An
1,SV2,Bình
2,SV3,Chương
3,SV4,Dũng


Ta cũng có thể kết nhiều bảng để được bảng kết quả đầy đủ thông tin hơn (khóa là MSSV).

In [29]:
pd.merge(stu_infor, pd.merge(score1, score2))

,MSSV,Tên sinh viên,Điểm1,Điểm2
0,SV1,An,7,4
1,SV3,Chương,9,8


Vì mỗi sinh viên (được xác định qua MSSV) chỉ xuất hiện tối đa một lần trong mỗi bảng trên nên phép kết các bảng này thường được gọi là kết **một-với-một** (one-to-one).

Giả sử ta có bảng thông tin các lớp học với khóa là mã lớp học (MLH)

In [30]:
class_infor = pd.DataFrame({'MLH': ['LP1', 'LP2'], 'Tên lớp': ["Lớp 1", "Lớp 2"]})
class_infor

,MLH,Tên lớp
0,LP1,Lớp 1
1,LP2,Lớp 2


và có bảng cho biết sinh viên thuộc lớp nào.

In [31]:
in_class = pd.DataFrame({'MSSV': ['SV1', 'SV2', 'SV3', 'SV4'], 'MLH': ["LP1", "LP1", "LP2", "LP2"]})
in_class

,MSSV,MLH
0,SV1,LP1
1,SV2,LP1
2,SV3,LP2
3,SV4,LP2


Bằng cách kết bảng học lớp với bảng thông tin sinh viên (trên khóa MLH), ta có bảng cho biết sinh viên học lớp tên gì.

In [32]:
pd.merge(in_class, class_infor)

,MSSV,MLH,Tên lớp
0,SV1,LP1,Lớp 1
1,SV2,LP1,Lớp 1
2,SV3,LP2,Lớp 2
3,SV4,LP2,Lớp 2


Vì mỗi lớp học (được xác định qua MLH) xuất hiện nhiều lần trong bảng học lớp (do có nhiều sinh viên học cùng lớp) và chỉ xuất hiện tối đa một lần trong bảng thông tin lớp học (nhiều sinh viên thuộc chung một lớp nhưng mỗi sinh viên chỉ thuộc một lớp) nên phép kết trên còn được gọi là **nhiều-với-một** (many-to-one).

Nếu để thứ tự ngược lại (trước, sau) ta có phép kết **một-với-nhiều** (one-to-many).

In [33]:
pd.merge(class_infor, in_class)

,MLH,Tên lớp,MSSV
0,LP1,Lớp 1,SV1
1,LP1,Lớp 1,SV2
2,LP2,Lớp 2,SV3
3,LP2,Lớp 2,SV4


Ta cũng có thể kết nhiều bảng để có thông tin đầy đủ.

In [34]:
stu_class = pd.merge(stu_infor, pd.merge(class_infor, in_class))
stu_class

,MSSV,Tên sinh viên,MLH,Tên lớp
0,SV1,An,LP1,Lớp 1
1,SV2,Bình,LP1,Lớp 1
2,SV3,Chương,LP2,Lớp 2
3,SV4,Dũng,LP2,Lớp 2


Ta có thể xóa bớt các cột để bảng kết quả "đỡ rườm rà".

In [35]:
stu_class.drop("MLH", axis=1)

,MSSV,Tên sinh viên,Tên lớp
0,SV1,An,Lớp 1
1,SV2,Bình,Lớp 1
2,SV3,Chương,Lớp 2
3,SV4,Dũng,Lớp 2


Giả sử ta có bảng thông tin môn học với khóa là mã môn học (MMH)

In [36]:
course_infor = pd.DataFrame({'MMH': ['MH1', 'MH2', 'MH3'], 'Tên môn': ["Môn 1", "Môn 2", "Môn 3"]})
course_infor

,MMH,Tên môn
0,MH1,Môn 1
1,MH2,Môn 2
2,MH3,Môn 3


và có bảng cho biết sinh viên đăng ký học môn nào.

In [37]:
learn_course = pd.DataFrame({'MSSV': ['SV1', 'SV1', 'SV2', 'SV3'], 'MMH': ["MH1", "MH2", "MH2", "MH1"]})
learn_course

,MSSV,MMH
0,SV1,MH1
1,SV1,MH2
2,SV2,MH2
3,SV3,MH1


Bằng cách kết các bảng ta có bảng cho biết sinh viên tên gì đăng ký học môn tên gì.

In [38]:
pd.merge(stu_infor, pd.merge(learn_course, course_infor))

,MSSV,Tên sinh viên,MMH,Tên môn
0,SV1,An,MH1,Môn 1
1,SV1,An,MH2,Môn 2
2,SV2,Bình,MH2,Môn 2
3,SV3,Chương,MH1,Môn 1


In [39]:
pd.merge(stu_infor, pd.merge(learn_course, course_infor)).drop(["MSSV", "MMH"], axis=1)

,Tên sinh viên,Tên môn
0,An,Môn 1
1,An,Môn 2
2,Bình,Môn 2
3,Chương,Môn 1


Vì mỗi sinh viên có thể đăng ký học nhiều môn học và mỗi môn học có thể có nhiều sinh viên đăng ký nên phép kết trên còn được gọi là **nhiều-với-nhiều** (many-to-many).

### <a name="ket_qua"/>Xác định cách giữ lại các dòng trong bảng kết quả

Mặc định, kết quả của thao tác kết chỉ giữ lại các dòng có giá trị khóa xuất hiện trong cả 2 bảng, còn gọi là phép **kết trong** (inner join).

Các cách kết khác có thể được xác định qua tham số `how` với các giá trị:
1. `inner`: giữ các dòng chung (giao)
1. `outer`: giữ tất cả các dòng (hợp)
1. `left`: giữ các dòng của bảng thứ nhất
1. `right`: giữ các dòng của bảng thứ hai

In [40]:
pd.merge(score1, score2, how="inner")

,MSSV,Điểm1,Điểm2
0,SV1,7,4
1,SV3,9,8


In [41]:
pd.merge(score1, score2, how="outer")

,MSSV,Điểm1,Điểm2
0,SV1,7.0,4.0
1,SV2,5.0,NaN
2,SV3,9.0,8.0
3,SV4,NaN,6.0


In [42]:
pd.merge(score1, score2, how="left")

,MSSV,Điểm1,Điểm2
0,SV1,7,4.0
1,SV2,5,NaN
2,SV3,9,8.0


In [43]:
pd.merge(score1, score2, how="right")

,MSSV,Điểm1,Điểm2
0,SV3,9.0,8
1,SV1,7.0,4
2,SV4,NaN,6


### <a name="key"/>Xác định khóa để kết bảng

Mặc định, khi kết các bảng, `merge` sẽ dùng các cột cùng tên làm khóa để kết. Tuy nhiên, ta có thể chỉ rõ khóa kết với các tham số khác nhau.

In [44]:
pd.merge(score1, score2)

,MSSV,Điểm1,Điểm2
0,SV1,7,4
1,SV3,9,8


In [45]:
pd.merge(score1, score2, on="MSSV")

,MSSV,Điểm1,Điểm2
0,SV1,7,4
1,SV3,9,8


In [46]:
pd.merge(score1, score2, left_on="MSSV", right_on="MSSV")

,MSSV,Điểm1,Điểm2
0,SV1,7,4
1,SV3,9,8


Thông thường, ta có thể dùng khóa làm chỉ mục cho các bảng.

In [47]:
score1

,MSSV,Điểm1
0,SV1,7
1,SV2,5
2,SV3,9


In [48]:
score1_idx = score1.drop("MSSV", axis=1)
score1_idx.index = score1.MSSV
score1_idx

,Điểm1
MSSV,
SV1,7
SV2,5
SV3,9


In [49]:
score2_idx = score2.drop("MSSV", axis=1)
score2_idx.index = score2.MSSV
score2_idx

,Điểm2
MSSV,
SV3,8
SV1,4
SV4,6


Khi đó ta có thể kết dựa trên index.

In [50]:
pd.merge(score1_idx, score2_idx, left_index=True, right_index=True, how="outer")

,Điểm1,Điểm2
MSSV,,
SV1,7.0,4.0
SV2,5.0,NaN
SV3,9.0,8.0
SV4,NaN,6.0


Tiện hơn, pandas cung cấp phương thức `join` để kết dựa trên index.

In [51]:
score1_idx.join(score2_idx)

,Điểm1,Điểm2
MSSV,,
SV1,7,4.0
SV2,5,NaN
SV3,9,8.0


In [52]:
score1_idx.join(score2_idx, how="outer")

,Điểm1,Điểm2
MSSV,,
SV1,7.0,4.0
SV2,5.0,NaN
SV3,9.0,8.0
SV4,NaN,6.0


In [53]:
stu_infor_idx = stu_infor.drop("MSSV", axis=1)
stu_infor_idx.index = stu_infor.MSSV
stu_infor_idx

,Tên sinh viên
MSSV,
SV1,An
SV2,Bình
SV3,Chương
SV4,Dũng


In [54]:
pd.merge(stu_infor, pd.merge(score1, score2), how="left")

,MSSV,Tên sinh viên,Điểm1,Điểm2
0,SV1,An,7.0,4.0
1,SV2,Bình,NaN,NaN
2,SV3,Chương,9.0,8.0
3,SV4,Dũng,NaN,NaN


In [55]:
stu_infor_idx.join(score1_idx.join(score2_idx))

,Tên sinh viên,Điểm1,Điểm2
MSSV,,,
SV1,An,7.0,4.0
SV2,Bình,5.0,NaN
SV3,Chương,9.0,8.0
SV4,Dũng,NaN,NaN


**Bài tập** Trên bảng dữ liệu `Iris Data Set` với tên `iris`

1. Thay cột class chuỗi thành số với các số mã cho nhóm tương ứng:
   - 0: setosa

   - 1: versicolour
   
   - 2: virginica

1. Tạo bảng `iris_class_name` chứa thông tin mã số ứng với nhãn lớp

1. Kết bảng `iris` với bảng `iris_class_name` để có lại thông tin nhãn lớp

1. Từ bảng `iris`, trích ra các dòng có sepal length lớn hơn trung bình và kết với bảng `iris_class_name` để cho ra kết quả với nhãn lớp

In [56]:
# TODO
